# Testing Pandas for EBD Processing

Note: this data is pulling from an unzipped txt EBD file in the "ebd" folder not included in the repository

## Resources

- [Pandas vs R Cheatsheet](https://datascientyst.com/pandas-vs-r-cheat-sheet/)
- [Combining/Merge/Axis with dataframes](https://www.geeksforgeeks.org/pandas/how-to-combine-two-dataframe-in-python-pandas/)

In [ ]:
import pandas as pd
import json
import numpy as np
import geopandas as gpd


# eBird Dataset file name
ebd_fn = 'ebd_US-NC_202101_202602_unv_smp_relMar-2026.txt'

# define data types
dtypes = {'LAST EDITED DATE' : str, 'COUNTRY' : str, 'COUNTRY CODE' : str,
'STATE' : str, 'STATE CODE' : str, 'COUNTY' : str, 'COUNTY CODE' : str,
'IBA CODE' : str, 'BCR CODE' : str, 'USFWS CODE' : str, 'ATLAS BLOCK' : str,
'LOCALITY' : str, 'LOCALITY ID' : str, 'LOCALITY TYPE' : str,
'LATITUDE' : float, 'LONGITUDE' : float, 'OBSERVATION DATE' : str,
'TIME OBSERVATIONS STARTED' : str, 'OBSERVER ID' : str, 
'OBSERVER ORCID ID' : str, 'SAMPLING EVENT IDENTIFIER' : str,
'OBSERVATION TYPE' : str, 'PROTOCOL NAME' : str, 'PROTOCOL CODE' : str,
'PROJECT NAMES' : object, 'PROJECT IDENTIFIERS' : object, 'DURATION MINUTES' : float,
'EFFORT DISTANCE KM' : str, 'EFFORT AREA HA' : str, 'NUMBER OBSERVERS' : float,
'ALL SPECIES REPORTED' : str, 'GROUP IDENTIFIER' : str, 'HAS MEDIA' : str,
'CHECKLIST COMMENTS' : str, 'GLOBAL UNIQUE IDENTIFIER' : str, 
'TAXONOMIC ORDER' : str, 'CATEGORY' : str, 'TAXON CONCEPT ID' : str,
'COMMON NAME' : str, 'SCIENTIFIC NAME' : str, 'SUBSPECIES COMMON NAME' : str,
'SUBSPECIES SCIENTIFIC NAME' : str, 'EXOTIC CODE' : str, 
'OBSERVATION COUNT' : str, 'BREEDING CODE' : str, 'BREEDING CATEGORY' : str,
'BEHAVIOR CODE' : str, 'AGE/SEX' : str, 'APPROVED' : str, 'REVIEWED' : str,
'REASON' : str, 'SPECIES COMMENTS' : str}

## Load the EBD into a Pandas DF

In [ ]:
## PRODUCTION
ebd_raw = pd.read_csv(f'./ebd/{ebd_fn}', sep='\t', dtype = dtypes)
print(f'Full ebd dimensions:{ebd_raw.shape}')
## END PRODUCTION

## Create Test Dataset

In [ ]:
# create subset for testing


### TEST
# ebd_raw = pd.read_csv("ebd_test.csv", dtype = dtypes, index_col = 0)

## save data
# ebd = pd.concat([ebd_raw.head(1000000), ebd_raw.tail(1000000)])
# ebd.to_csv("ebd_test.csv")
### END TEST

ebd = ebd_raw.fillna('')
print(ebd.info())


## Start Exploring

In [ ]:
rows_columns = ebd.shape
print(f'{rows_columns[0]:,d} rows, {rows_columns[1]} columns found')
# ebd['SAMPLING EVENT IDENTIFIER'].sort_values()

# remove spaces from colum names
ebd.columns = ebd.columns.str.replace(' ', '_')

# get column names
ebd_columns =  ebd.columns

# define observation fields
observation_fields = pd.Series([
  "GLOBAL_UNIQUE_IDENTIFIER",
  "TAXONOMIC_ORDER",
  "CATEGORY",
  "TAXON_CONCEPT_ID",
  "COMMON_NAME",
  "SCIENTIFIC_NAME",
  "SUBSPECIES_COMMON_NAME",
  "SUBSPECIES_SCIENTIFIC_NAME",
  "EXOTIC_CODE",
  "OBSERVATION_COUNT",
  "BREEDING_CODE",
  "BREEDING_CATEGORY",
  "BEHAVIOR_CODE",
  "AGE/SEX",
  "APPROVED",
  "REVIEWED",
  "REASON",
  "SPECIES_COMMENTS"
])

# get checklist fields
checklist_fields = pd.Series(ebd_columns[~ebd_columns.isin(observation_fields)])

# add SAMPLING EVENT IDENTIFIER to observations, remove Unamed column from checklist_fields
# observation_fields = pd.concat([observation_fields, pd.Series(['SAMPLING_EVENT_IDENTIFIER'])])
# checklist_fields = checklist_fields[checklist_fields != 'Unnamed: 52']

print(f'checklist fields:\n{list(checklist_fields)}')
print(f'observation fields:\n{list(observation_fields)}')

## Augment Checklist Data

### Get Block Info

In [ ]:
# get block layer

gpd_blocks = gpd.read_file("blocks.geojson")
gpd_blocks.shape

In [ ]:
# subset ebd for checklists, lat/lon only
ebd_checklists = (ebd.groupby([
    'SAMPLING_EVENT_IDENTIFIER',
    'LATITUDE',
    'LONGITUDE'
    ])
    .size()
    .to_frame("NUM_CHECKLISTS")
    .reset_index()
    )
# print(ebd_checklists.info())
# print(ebd_checklists.head())

# convert to geodataframe
gpd_checklists = gpd.GeoDataFrame(
    ebd_checklists,
    geometry = gpd.points_from_xy(
        ebd_checklists.LONGITUDE,
        ebd_checklists.LATITUDE
    ),
    crs = "EPSG:4326"
)

# spatial join with blocks to get block info
checklist_blocks = gpd_checklists.sjoin(
    gpd_blocks, how = 'left')

# print(gpd_blocks.head())
# print(list(checklist_blocks.columns))

# add priority block field
checklist_blocks['PRIORITY_BLOCK'] = np.where(
    checklist_blocks.block_type == "Priority", 1, 0
)

block_fields = checklist_blocks[['SAMPLING_EVENT_IDENTIFIER', 'ID_NCBA_BLOCK', 'ID_BLOCK_CODE', 'PRIORITY_BLOCK']]
block_fields = block_fields.fillna('').reset_index(drop = True)

# print(ncba_fields.info())
# print(ncba_fields.head())


## Group Observation Fields

Add SEI and group by this value.

In [ ]:
# make a list of obs fields plus SEI
sei = pd.Series(['SAMPLING_EVENT_IDENTIFIER'])
observation_fields = pd.concat([observation_fields, sei])
obs_only = ebd[observation_fields]

obs_grouped = (
    obs_only.groupby('SAMPLING_EVENT_IDENTIFIER')
    .apply(lambda x: x[list(observation_fields)].to_dict('records'))
    .reset_index()
    .rename(columns={0:'OBSERVATIONS'})
)

## Group Checklist Fields

Summarize each field.

In [ ]:
# # add fields
# ebd['YEAR'] = ebd['OBSERVATION_DATE'].str[:4].astype(int)
# ebd['MONTH'] = ebd['OBSERVATION_DATE'].str[5:7].astype(int)
# ebd['PROJECT_CODE'] = np.where(
#     ebd['PROJECT_NAMES'].str.contains('North Carolina Bird Atlas'),
#     'EBIRD_ATL_NC',
#     'EBIRD'
# )
# # ebd['ID_NCBA_BLOCK'] = np.where(
# #     ncba_fields[ncba_fields]
# # )
# # ebd['ID_BLOCK_CODE'] = ""
# # ebd['PRIORITY_BLOCK'] = ""
# ebd['NCBA_JULIAN_DAY'] = ""
# ebd['NCBA_WEEK'] = ""
# ebd['NCBA_SEASON'] = ""
# ebd['NCBA_QUARTER'] = ""
# ebd['NCBA_DATE_ADDED'] = "2026-04-27"
# ebd['NCBA_EBD_VER'] = "Mar-2026"
# ebd['NCBA_OBSERVER'] = ""

In [ ]:
# get checklists only, collapse fields that are repeats
check_only = (
    ebd[checklist_fields] # get only checklist fields
    .groupby('SAMPLING_EVENT_IDENTIFIER')
    .first() # get first item in grouping
    .reset_index()
    .iloc[:,:-1] # remove last column (unnamed - old index)
)
# print(check_only.info())
# print(check_only.head())

# add NCBA Fields
check_only['YEAR'] = check_only['OBSERVATION_DATE'].str[:4].astype(int)
check_only['MONTH'] = check_only['OBSERVATION_DATE'].str[5:7].astype(int)
check_only['PROJECT_CODE'] = np.where(
    check_only['PROJECT_NAMES'].str.contains('North Carolina Bird Atlas'),
    'EBIRD_ATL_NC',
    'EBIRD'
)

check_only['NCBA_JULIAN_DAY'] = (
    pd.to_datetime(check_only['OBSERVATION_DATE'])
    .dt.dayofyear
)
check_only['NCBA_WEEK'] = (
    pd.to_datetime(check_only['OBSERVATION_DATE'])
    .dt.isocalendar().week
)

# breeding_start = getJDay("2021-03-01") #day 60
# breeding_end = getJDay("2021-08-15") #day 227
# winter_start = getJDay("2021-11-01") #day 305
# winter_end = getJDay("2022-02-01") #day 32

check_only['NCBA_SEASON'] = np.where(
    # (check_only['NCBA_JULIAN_DAY'] >= 60),
    ((check_only['NCBA_JULIAN_DAY'] >= 60) & (check_only['NCBA_JULIAN_DAY'] <= 227)),
    'breeding',
    np.where(
        ((check_only['NCBA_JULIAN_DAY'] <= 32) | (check_only['NCBA_JULIAN_DAY'] >=305)),
        'wintering',
        'interim'
    )
)
# # ncba_fields['NCBA_QUARTER'] = ""
check_only['NCBA_DATE_ADDED'] = "2026-05-04"
check_only['NCBA_EBD_VER'] = "Mar-2026"

# add NCBA Volunteer field
from ebd_functions import id_observer_row
check_only['NCBA_OBSERVER'] = check_only.apply(
    id_observer_row,
    axis = 1
)
# add block and other NCBA fields
check_only = pd.merge(
    check_only,
    block_fields,
    how = 'left',
    on = 'SAMPLING_EVENT_IDENTIFIER'
)
check_only = check_only.drop(
    ["ID_NCBA_BLOCK", "ID_BLOCK_CODE", "PRIORITY_BLOCK"],
    axis = 'columns'
)
check_only.shape
# print(check_only.info())
# print(check_only.head())

## Join Observations and Checklists

In [ ]:

ebd_out = pd.merge(
    check_only,
    obs_grouped,
    how = 'inner',
    on = 'SAMPLING_EVENT_IDENTIFIER'
)

# print(ebd_out.info())
# print(ebd_out.head())


ebd_json = ebd_out.to_dict(orient='records')
# File for test.json
out_json = open(f"./ebd/test.json", 'w', encoding = 'utf-8-sig')
out_json.write(json.dumps(ebd_json))

# ebd_out.to_csv("ebd_out.csv")